# ════════════════════════════════════════
#  Movies Dataset — EDA Project
# ════════════════════════════════════════
# Dataset : Kaggle - The Movies Dataset
# Goal    : Analyze 45K+ movies trends
# Date    : 2026
# ════════════════════════════════════════


In [86]:
import pandas as pd
import numpy as np
print("libraries loaded")

libraries loaded


In [87]:
movies=pd.read_csv("movies_metadata.csv",low_memory=False)
ratings=pd.read_csv("ratings_small.csv")
print("Movies Shape : ",movies.shape)
print("Ratings shape : ",ratings.shape)

Movies Shape :  (45466, 24)
Ratings shape :  (100004, 4)


In [88]:
print(movies.dtypes)
print("_"*40)
print(movies.head())

adult                        str
belongs_to_collection        str
budget                       str
genres                       str
homepage                     str
id                           str
imdb_id                      str
original_language            str
original_title               str
overview                     str
popularity                   str
poster_path                  str
production_companies         str
production_countries         str
release_date                 str
revenue                  float64
runtime                  float64
spoken_languages             str
status                       str
tagline                      str
title                        str
video                     object
vote_average             float64
vote_count               float64
dtype: object
________________________________________
   adult                              belongs_to_collection    budget  \
0  False  {'id': 10194, 'name': 'Toy Story Collection', ...  30000000   
1  Fals

In [89]:
missing=movies.isnull().sum()
missing_pct=(missing/len(movies)*100).round(1)
missing_df=pd.DataFrame({
    "number_of_missings":missing,
    "percentage":missing_pct
})
print(missing_df[missing_df["number_of_missings"]>0].sort_values("percentage",ascending=False))

                       number_of_missings  percentage
belongs_to_collection               40972        90.1
homepage                            37684        82.9
tagline                             25054        55.1
overview                              954         2.1
poster_path                           386         0.8
runtime                               263         0.6
release_date                           87         0.2
status                                 87         0.2
imdb_id                                17         0.0
popularity                              5         0.0
original_language                      11         0.0
revenue                                 6         0.0
production_countries                    3         0.0
production_companies                    3         0.0
spoken_languages                        6         0.0
title                                   6         0.0
video                                   6         0.0
vote_average                

# cleaning data

In [90]:
cols_to_drop=[
    "belongs_to_collection","homepage","tagline","poster_path","adult","video"
]
movies.drop(columns=cols_to_drop,inplace=True,errors="ignore")
movies["budget"]=pd.to_numeric(movies["budget"],errors="coerce")
movies["popularity"]=pd.to_numeric(movies["popularity"],errors="coerce")
movies["release_date"]=pd.to_datetime(movies["release_date"],errors="coerce")

movies["year"]=movies["release_date"].dt.year
print("New shape : ",movies.shape)
print(movies[["budget","popularity","year"]].dtypes)

New shape :  (45466, 19)
budget        float64
popularity    float64
year          float64
dtype: object


# filtering 

In [91]:
print("Budget = 0   :", (movies["budget"] == 0).sum())
print("Revenue = 0  :", (movies["revenue"] == 0).sum())
print("Runtime = 0  :", (movies["runtime"] == 0).sum())

Budget = 0   : 36573
Revenue = 0  : 38052
Runtime = 0  : 1558


# from the previous step i conclude that the 0 values make the data unclean so i have to remove it and create a clean data set by put some criteria

In [92]:
movies_clean=movies[
    (movies["budget"]>100000) &
    (movies["revenue"]>100000) &
    (movies["vote_count"]>=50) &
    (movies["year"]>=1970) &
    (movies["year"]<=2017)
    ].copy()
print("Full dataset  : ",len(movies))
print("clean dataset : ",len(movies_clean))
print("Removed       : ",len(movies)-len(movies_clean))


Full dataset  :  45466
clean dataset :  4077
Removed       :  41389


# analysis

In [93]:
print("years range : ",int (movies_clean["year"].min())," -> ",int(movies_clean["year"].max()))
print("AVG rating : ",movies_clean["vote_average"].mean().round(2))
print("AVG budget : $ ",(movies_clean["budget"].mean()/1e6).round(2))
print("AVG revenue : $ ",(movies_clean["revenue"].mean()/1e6).round(1),"M")#convert the number into a million
print("AVG runtime : $ ",movies_clean["runtime"].mean().round(0),"min")

years range :  1970  ->  2017
AVG rating :  6.33
AVG budget : $  38.8
AVG revenue : $  115.3 M
AVG runtime : $  110.0 min


In [94]:
print("MIN Rating : ",movies_clean["vote_average"].min())
print("MAX Rating : ",movies_clean["vote_average"].max())
print("Average Rating : ",movies_clean["vote_average"].mean().round(2))
print("Median Rating : ",movies_clean["vote_average"].median())

MIN Rating :  2.8
MAX Rating :  9.1
Average Rating :  6.33
Median Rating :  6.4


In [95]:
print("MIN Budget : ",movies_clean["budget"].min())
print("MAX Budget : ",movies_clean["budget"].max())
print("Average Budget : ",(movies_clean["budget"].mean()/1e6).round(2),"M")
print("Median Budget : ",(movies_clean["budget"].median()/1e6).round(2),"M")

MIN Budget :  103000.0
MAX Budget :  380000000.0
Average Budget :  38.8 M
Median Budget :  25.0 M


In [96]:
print("MIN Revenue : ",movies_clean["revenue"].min())
print("MAX Revenue : ",movies_clean["revenue"].max())
print("Average Revenue : ",(movies_clean["revenue"].mean()/1e6).round(2),"M")
print("Median Revenue : ",(movies_clean["revenue"].median()/1e6).round(2),"M")

MIN Revenue :  100659.0
MAX Revenue :  2787965087.0
Average Revenue :  115.31 M
Median Revenue :  50.37 M


In [97]:
movies_clean["ROI"]=movies_clean["revenue"]/movies_clean["budget"]
print("MIN ROI : ",movies_clean["ROI"].min())
print("MAX ROI : ",movies_clean["ROI"].max())
print("Average ROI : ",movies_clean["ROI"].mean().round(2))
print("Median ROI : ",movies_clean["ROI"].median().round(2))

MIN ROI :  0.0051646
MAX ROI :  653.8461538461538
Average ROI :  4.69
Median ROI :  2.24


In [98]:
Top_ROI=movies_clean.sort_values("ROI",ascending=False).head(10)
print("the TOP 10 ROI films \n",Top_ROI[["title","budget","revenue","ROI"]])
Top_Revenue=movies_clean.sort_values("revenue",ascending=False).head(10)
print("__"*20)
print("\nTop_Revenue\n",Top_Revenue[["title","budget","revenue","ROI"]])

the TOP 10 ROI films 
                        title     budget      revenue         ROI
4316   The Way of the Dragon   130000.0   85000000.0  653.846154
9461              Open Water   130000.0   54667954.0  420.522723
3580                 Mad Max   400000.0  100000000.0  250.000000
1873               Halloween   300000.0   70000000.0  233.333333
3245       American Graffiti   777000.0  140000000.0  180.180180
11826                   Once   160000.0   20710513.0  129.440706
1845                   Rocky  1000000.0  117235147.0  117.235147
7733       Napoleon Dynamite   400000.0   46118097.0  115.295243
10660            Keeping Mum   169000.0   18564702.0  109.850308
5452     The Hills Have Eyes   230000.0   25000000.0  108.695652
________________________________________

Top_Revenue
                                               title       budget  \
14551                                        Avatar  237000000.0   
26555                  Star Wars: The Force Awakens  245000000.0   
163

In [99]:
import ast
import plotly.express as px
import plotly.figure_factory as ff
import plotly.io as pio

pio.renderers.default = "notebook"

movies_clean['genres'] = movies.loc[movies_clean.index, 'genres'].apply(
    lambda x: x if isinstance(x, list) else (ast.literal_eval(x) if isinstance(x, str) else [])
)
movies_clean['genre_name'] = movies_clean['genres'].apply(
    lambda g: g[0]['name'] if isinstance(g, list) and len(g) > 0 else None
)

fig1 = px.scatter(
    movies_clean,
    x="budget", y="revenue", size="ROI",
    hover_name="title",
    title="Budget vs Revenue  |  Bubble Size = ROI",
    labels={"budget": "Budget ($)", "revenue": "Revenue ($)"},
    color="ROI", color_continuous_scale="Viridis",
)

fig2 = px.histogram(
    movies_clean,
    x="vote_average", nbins=20,
    title="Distribution of Movie Ratings",
    labels={"vote_average": "Average Rating"},
    color_discrete_sequence=["#636EFA"],
)

genres_count = (
    movies_clean['genre_name']
    .dropna()
    .value_counts()
    .rename_axis('Genre')
    .reset_index(name='Count')
)
fig3 = px.bar(
    genres_count,
    x="Genre", y="Count",
    title="Top Movie Genres",
    color="Count", color_continuous_scale="Blues",
)

movies_exploded = movies_clean.explode('genres').copy()
movies_exploded['genre_label'] = movies_exploded['genres'].apply(
    lambda g: g['name'] if isinstance(g, dict) else None
)
fig4 = px.box(
    movies_exploded.dropna(subset=['genre_label']),
    x="genre_label", y="vote_average",
    title="Ratings Distribution by Genre",
    labels={"genre_label": "Genre", "vote_average": "Average Rating"},
)

corr = movies_clean[['budget', 'revenue', 'vote_average', 'ROI']].corr()
fig5 = ff.create_annotated_heatmap(
    z=corr.values,
    x=list(corr.columns),
    y=list(corr.columns),
    annotation_text=corr.round(2).values,
    colorscale='Blues',
)
fig5.update_layout(title="Correlation Heatmap")

top10 = movies_clean.nlargest(10, 'revenue')
fig6 = px.bar(
    top10,
    x="title", y="revenue",
    title="Top 10 Movies by Revenue",
    labels={"title": "Movie", "revenue": "Revenue ($)"},
    color="revenue", color_continuous_scale="Viridis",
)

dark_bg  = "#0d1117"
card_bg  = "#161b22"
text_col = "#c9d1d9"

for fig in [fig1, fig2, fig3, fig4, fig5, fig6]:
    fig.update_layout(
        paper_bgcolor=card_bg,
        plot_bgcolor=card_bg,
        font=dict(color=text_col),
        xaxis=dict(gridcolor="#30363d", linecolor="#30363d"),
        yaxis=dict(gridcolor="#30363d", linecolor="#30363d"),
    )

with open("dashboard.html", "w", encoding="utf-8") as f:
    f.write("<h1 style='font-family:Arial;text-align:center;color:#c9d1d9;background:#0d1117;padding:20px'>🎬 Movies Dashboard</h1>")
    for fig in [fig1, fig2, fig3, fig4, fig5, fig6]:
        f.write(fig.to_html(full_html=False, include_plotlyjs='cdn' if fig is fig1 else False))

print(" Dashboard saved → dashboard.html")

 Dashboard saved → dashboard.html
